In [2]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.models import Model
from keras.layers import InputLayer,Input
import random
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from tensorflow.keras.metrics import R2Score

SEED = 29

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [3]:
df = pd.read_parquet("train.parquet")

In [4]:
seqs = np.zeros((517,1000,32))
for i in range(517):
   seqs[i] = df[df["seq_ix"] == i].drop(columns= ["need_prediction","seq_ix","step_in_seq"]).values



In [5]:
seqs

array([[[-8.00821439e-02,  1.32456361e+00,  6.37730285e-01, ...,
          6.02751064e-01,  1.26518189e+00, -4.95939772e-01],
        [-1.24327372e+00,  3.98425198e-01, -1.81048479e+00, ...,
         -7.68425895e-02,  1.11189452e+00, -1.17143467e+00],
        [-1.24679579e-01,  1.22558017e+00,  1.39481886e-01, ...,
          8.56712288e-01,  1.20501645e+00, -4.17057281e-01],
        ...,
        [ 1.53859821e-01, -1.29540172e+00, -4.60807223e-01, ...,
         -8.37083087e-01, -8.97409380e-01,  3.81098386e-01],
        [ 1.16231196e-01,  1.04515735e+00,  2.72070336e-01, ...,
          6.23418442e-01,  9.17903219e-01,  4.72686298e-02],
        [ 3.22127546e-01,  7.32403833e-01,  5.43647191e-01, ...,
          6.65945148e-01,  5.63068400e-01,  5.25314208e-01]],

       [[-1.38531652e+00, -2.86290510e-01, -1.03217895e+00, ...,
          4.79279545e-01,  8.55656043e-01, -7.24305861e-01],
        [-6.29671662e-01, -1.54513467e+00, -9.21040809e-01, ...,
          1.08662669e+00, -2.41289933e

In [17]:
input_to_encoder_seqs = np.zeros((517,100,32))
for i in range(517):
   input_to_encoder_seqs[i] = seqs[i][0:100][:]


In [18]:
output_of_decoder_seqs = np.zeros((517,899,32))
for i in range(517):
   output_of_decoder_seqs[i] =  seqs[i][100:999][:]


In [19]:
seqs.max()

np.float64(5.19933758270342)

In [20]:
output_of_decoder_seqs

array([[[-0.04683886,  1.17929951,  0.21538239, ...,  0.66755463,
          1.11624375, -0.21616168],
        [ 0.35021217,  0.69358216,  0.57892525, ...,  0.62009971,
          0.5350674 ,  0.58963339],
        [-0.85833891, -0.24064856, -1.16566933, ..., -0.25065828,
          0.64747832, -0.66328607],
        ...,
        [ 1.19705128, -0.10438045,  1.01902289, ..., -0.4235358 ,
         -0.92590085,  1.97733601],
        [ 0.15385982, -1.29540172, -0.46080722, ..., -0.83708309,
         -0.89740938,  0.38109839],
        [ 0.1162312 ,  1.04515735,  0.27207034, ...,  0.62341844,
          0.91790322,  0.04726863]],

       [[ 1.62614553, -0.31321478,  0.68528999, ..., -0.5033426 ,
         -1.25302335,  2.45870023],
        [-0.20888137, -1.5849149 , -0.78053663, ...,  0.25147081,
         -0.67354582,  0.46727728],
        [-1.35799945, -0.28711591, -1.16287408, ..., -1.05469702,
          0.53837614, -0.07228047],
        ...,
        [-1.70759473, -0.09285136, -1.20304512, ..., -

In [21]:
import tensorflow.keras.backend as K
def r2_keras(y_true, y_pred):
    """
    Custom R2-score metric for 3D tensors (batch, time_steps, features).
    Flattens the time steps dimension before calculating R2.
    """
    # Reshape tensors from (batch, time_steps, features) to (batch * time_steps, features)
    y_true_reshaped = K.reshape(y_true, (-1, K.shape(y_true)[-1]))
    y_pred_reshaped = K.reshape(y_pred, (-1, K.shape(y_pred)[-1]))

    # Calculate Sum of Squares Residual (SS_res)
    SS_res = K.sum(K.square(y_true_reshaped - y_pred_reshaped))

    # Calculate Total Sum of Squares (SS_tot)
    # Mean is calculated across all flattened samples
    SS_tot = K.sum(K.square(y_true_reshaped - K.mean(y_true_reshaped)))

    # Calculate R2 score, add epsilon to prevent division by zero
    r2 = 1 - SS_res / (SS_tot + K.epsilon())

    return r2

In [42]:
class AutoEncoder(Model):
  def __init__(self,input_shape):
    self.num_neurons = [40,40,40]


    self.filters = 40
    self.kernel_size = 5
    
    self.enc_dims = 64
    self.decoder_neurons = self.num_neurons[-1]
    self.to_predict = 899
    # self.encoder_layers = []
    # self.decoder_layers = []
    super(AutoEncoder,self).__init__()
    

    self.conv_layer = layers.Conv1D(self.filters,self.kernel_size,padding = "causal",activation = "relu")

    self.encoder_lstm_1 = layers.LSTM(self.num_neurons[0],
                                          return_sequences=True,
                                          name="encoder_lstm_1",
                                          recurrent_dropout = 0.3) 
    self.encoder_lstm_middle = layers.LSTM(self.num_neurons[1],
                                          return_sequences=True,
                                          name="encoder_lstm_middle")
    self.projector_layer = layers.Dense(self.num_neurons[1],activation="linear",name = "projector")
    
    self.encoder_lstm_2 = layers.LSTM(self.num_neurons[2],
                                          return_sequences=True,
                                          return_state=True,
                                          name="encoder_lstm_2")

    self.decoder_lstm = layers.LSTM(self.decoder_neurons,
                                         return_sequences=True,
                                         return_state=True,
                                         name="decoder_lstm")
    self.scaler = layers.Normalization(name = "scaler")
    self.norm_1 = layers.LayerNormalization(name = "norm1")
    self.norm_2 = layers.LayerNormalization(name = "norm2")
    self.norm_3 = layers.LayerNormalization(name = "norm3")
    self.norm_4 = layers.LayerNormalization(name = "norm4")

    self.DenseQ = layers.Dense(self.enc_dims,name = "Query_Dense")
    self.DenseK = layers.Dense(self.enc_dims,name = "Query_Key")
    self.DenseV = layers.Dense(self.enc_dims,name = "Query_Value")
    self.dropout = layers.Dropout(0.3)

    # self.FFn_neurons = [24,36,self.enc_dims]

    # self.FFn_Dense1 = layers.Dense()

    num_heads = 4
    self.self_attention_layer = layers.MultiHeadAttention(name = "self_attention",num_heads = num_heads,key_dim = self.enc_dims//num_heads)
    self.cross_attention_layer = layers.MultiHeadAttention(name="attention",num_heads = num_heads,key_dim= self.enc_dims//num_heads)
    
    self.concat_layer = layers.Concatenate(axis=-1, name="concat_attention")
    self.output_dense_layer = layers.Dense(input_shape[1], name="output_dense",activation="linear") 
    self.output_time_dist = layers.TimeDistributed(self.output_dense_layer, name="output_time_distributed") 


   # print("yes issue\n")
    inputs = tf.keras.Input(input_shape,name = "init_input")  
   # print("no issue\n")
    #inputs = self.scaler(inputs)
    temp = self.conv_layer(inputs)
    x1 = self.encoder_lstm_1(temp)
    x = self.encoder_lstm_middle(x1)
    x1_proj = self.projector_layer(x1)
    x = layers.Add(name = "skip_conn")([x1_proj,x])
    x = self.norm_1(x)
    
    lstm_out , h , c = self.encoder_lstm_2(x)
    states = [h,c]

    key = self.DenseK(lstm_out)
    query = self.DenseQ(lstm_out)
    val = self.DenseV(lstm_out)

    attention_context_for_enc = self.self_attention_layer(query =query , value = val,key =key)
    encoder_out = self.concat_layer([attention_context_for_enc,lstm_out])
    encoder_out = self.norm_2(encoder_out)


    self.encoder_model = Model(inputs = inputs,outputs = [encoder_out] + states , name = "ENCODER")

    target = tf.keras.Input(shape = (self.to_predict,input_shape[1]))
    decoder_lstm_outputs, _, _ = self.decoder_lstm(target,initial_state=states)



    attention_context = self.cross_attention_layer(decoder_lstm_outputs, encoder_out,encoder_out)
    decoder_combined_context = self.concat_layer([attention_context, decoder_lstm_outputs])
    decoder_combined_context = self.norm_3(decoder_combined_context)
    decoder_combined_context = self.dropout(decoder_combined_context)
    decoder_pred_outputs = self.output_time_dist(decoder_combined_context)


    self.training_model = Model(inputs=[inputs, target],outputs=decoder_pred_outputs)

    #----------------------------------------------------------------------------------------------------
    inf_decoder_input_prev = Input(shape=(1, input_shape[1]), name="inf_decoder_input") # Previous prediction
    inf_decoder_state_h = Input(shape=(self.decoder_neurons,), name="inf_state_h")          # Previous h state
    inf_decoder_state_c = Input(shape=(self.decoder_neurons,), name="inf_state_c")          # Previous c state
    inf_decoder_states_prev = [inf_decoder_state_h, inf_decoder_state_c]                 # Combined previous states
    inf_encoder_outputs = Input(shape=(input_shape[0], self.num_neurons[-1]+self.enc_dims), name="inf_encoder_outputs") # ALL encoder outputs for attention

        # Connect layers for one step using shared layer instances:
    inf_dec_lstm_out, inf_state_h_out, inf_state_c_out = self.decoder_lstm(
        inf_decoder_input_prev, initial_state=inf_decoder_states_prev # Initialize with previous states
    )
    inf_attention_context = self.cross_attention_layer(inf_dec_lstm_out, inf_encoder_outputs,inf_encoder_outputs) # Calculate attention for this step
    inf_combined_context = self.concat_layer([inf_attention_context, inf_dec_lstm_out])    # Combine
    inf_decoder_pred = self.output_time_dist(inf_combined_context)                       # Predict (shape=(batch, 1, features))

        # Define the single-step inference decoder model
    self.inf_decoder_model = Model(
        inputs=[inf_decoder_input_prev, inf_decoder_states_prev, inf_encoder_outputs],
        outputs=[inf_decoder_pred] + [inf_state_h_out, inf_state_c_out], # Output prediction AND next states
        name="inf_decoder"
    )      

    
                            
  

  def train(self,X_train,y_train):
    # print("Adapting scaler...")
    # self.scaler.adapt(X_train)
    # print("Scaler adapted.")
    last_known_step = X_train[:, -1:, :] 
    reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_r2_keras',  
        factor=0.5,          
        patience=2,          
        verbose=1,           
        mode='max',         
        min_delta=0.002,      
        cooldown=2,          
        min_lr=0.000002        
    )
    future_inputs = y_train[:, :-1, :]

    # learning_scheduler = tf.keras.optimizers.schedules.ExponentialDecay(
    #     0.01,
    #     decay_steps=10000,
    #     decay_rate=0.96,
    #     staircase=True)
    
    # 3. CREATE THE NEW DECODER INPUT VARIABLE
    # This is the "shifted" sequence for teacher forcing
    # Shape: (batch_size, 899, 32)
    decoder_input_data = np.concatenate([last_known_step, future_inputs], axis=1)
    
    # 4. THE TARGET IS JUST THE ORIGINAL Y_TRAIN
    decoder_target_data = y_train
    optim = tf.keras.optimizers.AdamW(learning_rate=0.02,clipnorm = 1.0)
    self.training_model.compile(optimizer = optim,loss= "mse",metrics=[r2_keras])
    self.training_model.summary()
    self.training_model.fit([X_train,decoder_input_data],
                            y_train,batch_size = 32, 
                            epochs = 55,
                            verbose = True, 
                            validation_split = 0.2,
                            callbacks = [reduce_lr]
                            )
    self.training_model.save_weights("C:/CodingWorkspace/Python/WunderRNN/AE_wts_diffs.weights.h5")



In [43]:
autoenc = AutoEncoder([100,32])
autoenc.train(X_train = input_to_encoder_seqs,y_train=output_of_decoder_seqs)

Model: "functional_11"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ init_input          │ (None, 100, 32)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_11 (Conv1D)  │ (None, 100, 40)   │      6,440 │ init_input[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm_1      │ (None, 100, 40)   │     12,960 │ conv1d_11[0][0]   │
│ (LSTM)              │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ projector (Dense)   │ (None, 100, 40)   │      1,640 │ encoder_lstm_1[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm_middle │ (None, 100, 40)   │     12,960 │ encoder_lstm_1[0… │
│ (LSTM)              │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ skip_conn (Add)     │ (None, 100, 40)   │          0 │ projector[0][0],  │
│                     │                   │            │ encoder_lstm_mid… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ norm1               │ (None, 100, 40)   │         80 │ skip_conn[0][0]   │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm_2      │ [(None, 100, 40), │     12,960 │ norm1[0][0]       │
│ (LSTM)              │ (None, 40),       │            │                   │
│                     │ (None, 40)]       │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Query_Key (Dense)   │ (None, 100, 64)   │      2,624 │ encoder_lstm_2[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Query_Dense (Dense) │ (None, 100, 64)   │      2,624 │ encoder_lstm_2[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Query_Value (Dense) │ (None, 100, 64)   │      2,624 │ encoder_lstm_2[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ self_attention      │ (None, 100, 64)   │     16,640 │ Query_Key[0][0],  │
│ (MultiHeadAttentio… │                   │            │ Query_Dense[0][0… │
│                     │                   │            │ Query_Value[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concat_attention    │ (None, 1, 80)     │          0 │ self_attention[0… │
│ (Concatenate)       │                   │            │ encoder_lstm_2[0… │
│                     │                   │            │ attention[0][0],  │
│                     │                   │            │ decoder_lstm[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_11      │ (None, 899, 32)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm (LSTM) │ [(None, 1, 40),   │     11,680 │ input_layer_11[0… │
│                     │ (None, 40),       │            │ encoder_lstm_2[0… │
│                     │ (None, 40)]       │            │ encoder_lstm_2[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ norm2               │ (None, 100, 104)  │        208 │ concat_attention… │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention           │ (None, 1, 40)     │     18,664 │ decoder_lstm[0][

 Total params: 104,856 (409.59 KB)

 Trainable params: 104,856 (409.59 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/55
13/13 ━━━━━━━━━━━━━━━━━━━━ 41s 1s/step - loss: 1.8789 - r2_keras: -0.8897 - val_loss: 1.0166 - val_r2_keras: -0.0156 - learning_rate: 0.0200
Epoch 2/55
13/13 ━━━━━━━━━━━━━━━━━━━━ 9s 713ms/step - loss: 1.0273 - r2_keras: -0.0422 - val_loss: 0.9665 - val_r2_keras: 0.0323 - learning_rate: 0.0200
Epoch 3/55
13/13 ━━━━━━━━━━━━━━━━━━━━ 9s 696ms/step - loss: 0.9454 - r2_keras: 0.0403 - val_loss: 0.9347 - val_r2_keras: 0.0627 - learning_rate: 0.0200
Epoch 4/55
13/13 ━━━━━━━━━━━━━━━━━━━━ 9s 695ms/step - loss: 0.8924 - r2_keras: 0.0946 - val_loss: 0.8594 - val_r2_keras: 0.1335 - learning_rate: 0.0200
Epoch 5/55
13/13 ━━━━━━━━━━━━━━━━━━━━ 9s 690ms/step - loss: 0.8237 - r2_keras: 0.1642 - val_loss: 0.8125 - val_r2_keras: 0.1770 - learning_rate: 0.0200
Epoch 6/55
13/13 ━━━━━━━━━━━━━━━━━━━━ 9s 717ms/step - loss: 0.7781 - r2_keras: 0.2102 - val_loss: 0.7738 - val_r2_keras: 0.2142 - learning_rate: 0.0200
Epoch 7/55
13/13 ━━━━━━━━━━━━━━━━━━━━ 9s 682ms/step - loss: 0.7447 - r2_keras: 0.2439 -